# 01: Data Download

### Data Download for Clinical, Microenvironment, Single Nucleotide Variation, Copy Number Alteration, and Expression data
1. Go to https://portal.gdc.cancer.gov/analysis_page?app=CohortBuilder&tab=general
    - Select General -> Program -> TCGA
    - Select Demographic -> Gender -> female
2. Go to "Repository" with the previous selections still active.
2. Make the following selections: 
    - Experimental Strategy -> RNA-Seq
    - Access -> open
    - Tumor Descriptor -> primary + not applicable
3. "Add All Files to Cart". Go to "Cart". 
    - Download Associated Data, place in the `data_raw/female` folder, and unzip.
        - Clinical: TSV
        - Biospecimen: TSV
        - Sample Sheet
    - Download Cart -> Manifest, then place in the `data_raw/female/rnaseq` folder.
5. "Remove from Cart" -> "All Files"
6. In Repository, make the following selections:
    - Data Category -> copy number variation
    - Data Type -> Allele-specific Copy Number Segment
    - Workflow Type -> ASCAT3
    - Access -> open
    - Tissue Type -> tumor
7. "Add All Files to Cart". Go to "Cart". 
    - Download Cart -> Manifest, then place in the `data_raw/female/segments` folder.
8. "Remove from Cart" -> "All Files"
9. In Repository, make the following selections:
    - Data Category -> simple nucleotide variation
    - Access -> open
    - Tissue Type -> tumor
    - Tumor Descriptor -> primary
7. "Add All Files to Cart". Go to "Cart". 
    - Download Cart -> Manifest, then place in the `data_raw/female/snv` folder.
6. Go back to Cohort Builder, change Demographic from female to male.
5. Repeat from step 2, placing data in `data_raw/male`.
6. Use the GDC client CLI to download rnaseq files from manifests

```bash
cd data_raw/female/rnaseq
gdc-client download -m <your_manifest>.txt
```

Repeat for all manifests

In [10]:
import os
import pandas as pd
import numpy as np

# Replace these paths to run for both the "unknown" and "other" downloads
in_dir = 'data_raw/female'
out_dir = 'data_intermediate/female'
os.makedirs(out_dir, exist_ok=True)

# Replace these with your folder names
tcga_cases_tsv = 'data_raw/cases.tsv'
tcga_clinical_tsv = os.path.join(in_dir, 'clinical.cart.2025-02-03/clinical.tsv')
tcga_sample_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-02-03/sample.tsv')
tcga_slide_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-02-03/slide.tsv')
tcga_file_tsv = os.path.join(in_dir, 'gdc_sample_sheet.2025-02-03.tsv')

In [11]:
tcga_clinical_df = pd.read_csv(tcga_clinical_tsv, sep='\t')
tcga_sample_df = pd.read_csv(tcga_sample_tsv, sep='\t')
tcga_slide_df = pd.read_csv(tcga_slide_tsv, sep='\t')
tcga_file_df = pd.read_csv(tcga_file_tsv, sep='\t')

/var/folders/_6/468mk1cx3cb080z2r5gq0k_r0000gn/T/ipykernel_80740/2272134295.py:1: DtypeWarning: Columns (4,57) have mixed types. Specify dtype option on import or set low_memory=False.
  tcga_clinical_df = pd.read_csv(tcga_clinical_tsv, sep='\t')


In [12]:
selected_clinical_cols = [
    'case_id',
    'case_submitter_id',
    'project_id',
    'age_at_diagnosis',
    'gender',
    'year_of_birth',
    'race',
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
selected_survival_cols = [
    'case_id',
    'case_submitter_id',
    'vital_status',
    'days_to_death',
    'days_to_last_follow_up',
]
selected_sample_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'days_to_collection',
    'sample_type',
]
selected_slide_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'slide_submitter_id',
    'percent_neutrophil_infiltration',
    'percent_monocyte_infiltration',
    'percent_normal_cells',
    'percent_tumor_nuclei',
    'percent_lymphocyte_infiltration',
    'percent_stromal_cells',
    'percent_tumor_cells',
]
disease_names = {
    "LAML": "Acute Myeloid Leukemia",
    "ACC": "Adrenocortical carcinoma",
    "BLCA": "Bladder Urothelial Carcinoma",
    "LGG": "Brain Lower Grade Glioma",
    "BRCA": "Breast invasive carcinoma",
    "CESC": "Cervical squamous cell carcinoma and endocervical adenocarcinoma",
    "CHOL": "Cholangiocarcinoma",
    "LCML": "Chronic Myelogenous Leukemia",
    "COAD": "Colon adenocarcinoma",
    "ESCA": "Esophageal carcinoma",
    "GBM": "Glioblastoma multiforme",
    "HNSC": "Head and Neck squamous cell carcinoma",
    "KICH": "Kidney Chromophobe",
    "KIRC": "Kidney renal clear cell carcinoma",
    "KIRP": "Kidney renal papillary cell carcinoma",
    "LIHC": "Liver hepatocellular carcinoma",
    "LUAD": "Lung adenocarcinoma",
    "LUSC": "Lung squamous cell carcinoma",
    "DLBC": "Lymphoid Neoplasm Diffuse Large B-cell Lymphoma",
    "MESO": "Mesothelioma",
    "MISC": "Miscellaneous",
    "OV": "Ovarian serous cystadenocarcinoma",
    "PAAD": "Pancreatic adenocarcinoma",
    "PCPG": "Pheochromocytoma and Paraganglioma",
    "PRAD": "Prostate adenocarcinoma",
    "READ": "Rectum adenocarcinoma",
    "SARC": "Sarcoma",
    "SKCM": "Skin Cutaneous Melanoma",
    "STAD": "Stomach adenocarcinoma",
    "TGCT": "Testicular Germ Cell Tumors",
    "THYM": "Thymoma",
    "THCA": "Thyroid carcinoma",
    "UCS": "Uterine Carcinosarcoma",
    "UCEC": "Uterine Corpus Endometrial Carcinoma",
    "UVM": "Uveal Melanoma",
}
primary_sites = {
    'LAML': 'Blood',
    'ACC': 'Adrenal Gland',
    'BLCA': 'Bladder',
    'LGG': 'Brain',
    'BRCA': 'Breast',
    'CESC': 'Cervix',
    'CHOL': 'Bile Duct',
    'LCML': 'Blood',
    'COAD': 'Colorectal',
    'ESCA': 'Esophagus',
    'GBM': 'Brain',
    'HNSC': 'Head and Neck',
    'KICH': 'Kidney',
    'KIRC': 'Kidney',
    'KIRP': 'Kidney',
    'LIHC': 'Liver',
    'LUAD': 'Lung',
    'LUSC': 'Lung',
    'DLBC': 'Lymph Nodes',
    'MESO': 'Pleura',
    'MISC': 'Miscellanteous',
    'OV': 'Ovary',
    'PAAD': 'Pancreas',
    'PCPG': 'Adrenal Gland',
    'PRAD': 'Prostate',
    'READ': 'Colorectal',
    'SARC': 'Soft Tissue',
    'SKCM': 'Skin',
    'STAD': 'Stomach',
    'TGCT': 'Testis',
    'THYM': 'Thymus',
    'THCA': 'Thyroid',
    'UCS': 'Uterus',
    'UCEC': 'Uterus',
    'UVM': 'Eye'
}

In [13]:
print('SURVIVAL')
tcga_survival_df = tcga_clinical_df[selected_survival_cols]
display(tcga_survival_df.head())
print('CLINICAL')
tcga_clinical_df = tcga_clinical_df[selected_clinical_cols]
display(tcga_clinical_df.head())
print('SAMPLE')
tcga_sample_df = tcga_sample_df[selected_sample_cols]
display(tcga_sample_df.head())
print('SLIDE')
tcga_slide_df = tcga_slide_df[selected_slide_cols]
display(tcga_slide_df.head())
print('RNASEQ FILES')
tcga_file_df.drop_duplicates(subset='Sample ID', inplace=True, keep='last')
display(tcga_file_df.head())

SURVIVAL


,case_id,case_submitter_id,vital_status,days_to_death,days_to_last_follow_up
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,Alive,'--,'--
1,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,Alive,'--,'--
2,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,Alive,'--,'--
3,001e0309-9c50-42b0-9e38-347883ee2cd3,TCGA-D1-A17N,Alive,'--,'--
4,001e0309-9c50-42b0-9e38-347883ee2cd3,TCGA-D1-A17N,Alive,'--,'--


CLINICAL


,case_id,case_submitter_id,project_id,age_at_diagnosis,gender,year_of_birth,race,ajcc_pathologic_stage,ajcc_clinical_stage,ann_arbor_clinical_stage,figo_stage
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,TCGA-BRCA,22279,female,'--,white,Stage IA,'--,'--,'--
1,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,TCGA-BRCA,22279,female,'--,white,Stage IA,'--,'--,'--
2,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,TCGA-BRCA,22279,female,'--,white,Stage IA,'--,'--,'--
3,001e0309-9c50-42b0-9e38-347883ee2cd3,TCGA-D1-A17N,TCGA-UCEC,17045,female,'--,not reported,'--,'--,'--,Stage IA
4,001e0309-9c50-42b0-9e38-347883ee2cd3,TCGA-D1-A17N,TCGA-UCEC,17045,female,'--,not reported,'--,'--,'--,Stage IA


SAMPLE


,case_id,case_submitter_id,sample_id,sample_submitter_id,days_to_collection,sample_type
0,6a2155b3-915c-41b1-b730-32e55e0c1fbd,TCGA-GC-A3RD,1c60f60e-a3c6-43a1-a56e-ba68801d2d90,TCGA-GC-A3RD-01A,77,Primary Tumor
1,6a2155b3-915c-41b1-b730-32e55e0c1fbd,TCGA-GC-A3RD,623992fa-e316-4aef-9b0e-6c7eaa5121a5,TCGA-GC-A3RD-10B,172,Blood Derived Normal
2,6a2155b3-915c-41b1-b730-32e55e0c1fbd,TCGA-GC-A3RD,a7103730-93f8-4cea-bbb7-3954a5491eaa,TCGA-GC-A3RD-01Z,'--,Primary Tumor
3,cb695fdd-381e-4c00-836e-0ba27e32176b,TCGA-VS-A8QC,047b9b65-ae3a-47a9-9cd5-3bbe05471c8f,TCGA-VS-A8QC-01A,1768,Primary Tumor
4,cb695fdd-381e-4c00-836e-0ba27e32176b,TCGA-VS-A8QC,6bcd780e-ff97-49fb-9084-287c61db05e5,TCGA-VS-A8QC-10A,1768,Blood Derived Normal


SLIDE


,case_id,case_submitter_id,sample_id,sample_submitter_id,slide_submitter_id,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells
0,6a2155b3-915c-41b1-b730-32e55e0c1fbd,TCGA-GC-A3RD,1c60f60e-a3c6-43a1-a56e-ba68801d2d90,TCGA-GC-A3RD-01A,TCGA-GC-A3RD-01A-01-TS1,0.0,0.0,0.0,90.0,2.0,0.0,100.0
1,6a2155b3-915c-41b1-b730-32e55e0c1fbd,TCGA-GC-A3RD,a7103730-93f8-4cea-bbb7-3954a5491eaa,TCGA-GC-A3RD-01Z,TCGA-GC-A3RD-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--
2,cb695fdd-381e-4c00-836e-0ba27e32176b,TCGA-VS-A8QC,047b9b65-ae3a-47a9-9cd5-3bbe05471c8f,TCGA-VS-A8QC-01A,TCGA-VS-A8QC-01A-01-TS1,20.0,20.0,0.0,70.0,40.0,25.0,70.0
3,cb695fdd-381e-4c00-836e-0ba27e32176b,TCGA-VS-A8QC,98bf5752-fba7-40e1-869b-1857f0ece899,TCGA-VS-A8QC-01Z,TCGA-VS-A8QC-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--
4,257dc623-41e1-4fa3-859c-c787db10bd60,TCGA-IS-A3KA,01412943-d123-4c80-b258-850785d5335e,TCGA-IS-A3KA-01Z,TCGA-IS-A3KA-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--


RNASEQ FILES


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Sample Type
0,50fd12e8-71e9-40e9-8752-33cc2b537000,44d6fd71-0853-48e6-82c8-c4c1978278aa.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-HNSC,TCGA-CR-7371,TCGA-CR-7371-01A,Primary Tumor
1,00493cfb-cfcf-4994-b4f4-3f1ff94e21f8,6d1ffd0c-ba13-463e-b5c4-20d2dab6294d.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-HNSC,TCGA-CV-A463,TCGA-CV-A463-01A,Primary Tumor
2,867a2368-8027-45d1-b7b2-5a6f5523640f,e9952261-abcb-419f-9889-d98fe970cf20.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-HNSC,TCGA-CR-7374,TCGA-CR-7374-01A,Primary Tumor
4,56e9a740-ee72-4c10-a90d-921034141b42,2b4652fb-eef5-427e-b4f0-24164c042e52.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUAD,TCGA-44-2665,TCGA-44-2665-11A,Solid Tissue Normal
5,fc732523-d387-4a1d-ab81-34e44e869bac,f7beaa9b-98ae-40bd-aadc-e69688499170.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUAD,TCGA-44-2665,TCGA-44-2665-01B,Primary Tumor


In [14]:
# Compile clinical covariates
my_clinical_df = tcga_clinical_df.copy()
my_clinical_df['disease_type'] = tcga_clinical_df['project_id'].map(lambda s: s.split('-')[1])
my_clinical_df.drop(columns=['project_id', 'case_submitter_id'], inplace=True)
my_clinical_df['disease_name'] = my_clinical_df['disease_type'].map(disease_names)
my_clinical_df['primary_site'] = my_clinical_df['disease_type'].map(primary_sites)
my_clinical_df.drop_duplicates(subset='case_id', inplace=True)
my_clinical_df.replace("'--", None, inplace=True)

stage_features = [
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
stages = []
for i, (ajcc_path, ajcc_clin, ann, figo) in my_clinical_df[stage_features].iterrows():
    def get_stage(s):
        if "IV" in s:
            return 4.
        elif "III" in s:
            return 3.
        elif "II" in s:
            return 2.
        elif "I" in s:
            return 1.
        else:
            return None
    if ajcc_path is not None:
        stage = get_stage(ajcc_path)
    elif ajcc_clin is not None:
        stage = get_stage(ajcc_clin)
    elif ann is not None:
        stage = get_stage(ann)
    elif figo is not None:
        stage = get_stage(figo)
    else:
        stage = None
    stages.append(stage)
my_clinical_df['stage'] = stages 
my_clinical_df.drop(columns=stage_features, inplace=True)

my_clinical_df.head()

,case_id,age_at_diagnosis,gender,year_of_birth,race,disease_type,disease_name,primary_site,stage
0,001cef41-ff86-4d3f-a140-a647ac4b10a1,22279,female,None,white,BRCA,Breast invasive carcinoma,Breast,1.0
3,001e0309-9c50-42b0-9e38-347883ee2cd3,17045,female,None,not reported,UCEC,Uterine Corpus Endometrial Carcinoma,Uterus,1.0
7,0020317d-d10e-4e75-8fa6-7c1bdcdee471,24025,female,None,not reported,SARC,Sarcoma,Soft Tissue,NaN
9,002724fa-7051-49fa-9c58-4bcb7eba4ac6,26666,female,None,white,UCEC,Uterine Corpus Endometrial Carcinoma,Uterus,3.0
19,0030a28c-81aa-44b0-8be0-b35e1dcbf98c,22454,female,None,asian,DLBC,Lymphoid Neoplasm Diffuse Large B-cell Lymphoma,Lymph Nodes,1.0


In [15]:
# Compile sample covariates and merge with clinical covariates
# Only use the first tumor and healthy sample from each patient
my_sample_df = tcga_sample_df[tcga_sample_df['sample_submitter_id'].map(lambda s: '-11A' in s or '-01A' in s)]
my_sample_df = my_sample_df.merge(tcga_slide_df.drop(columns=['case_id', 'case_submitter_id', 'sample_submitter_id', 'slide_submitter_id']), on='sample_id')
my_sample_df.drop_duplicates(subset='sample_id', inplace=True)  # Multiple slides, just take the first
my_sample_df.replace("'--", None, inplace=True)

# Merge patient-level clinical covariates
my_sample_df = my_sample_df.merge(my_clinical_df, on='case_id', how='left')
my_sample_df['sample_id'] = my_sample_df['sample_submitter_id']
my_sample_df['submitter_id'] = my_sample_df['case_submitter_id']
my_sample_df.drop(columns=['case_id', 'case_submitter_id', 'sample_submitter_id'], inplace=True)

my_sample_df.to_csv(os.path.join(out_dir, 'clinical_covariates.csv'), index=False)
my_sample_df.head()

,sample_id,days_to_collection,sample_type,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells,age_at_diagnosis,gender,year_of_birth,race,disease_type,disease_name,primary_site,stage,submitter_id
0,TCGA-GC-A3RD-01A,77,Primary Tumor,0.0,0.0,0.0,90.0,2.0,0.0,100.0,30529,female,None,white,BLCA,Bladder Urothelial Carcinoma,Bladder,3.0,TCGA-GC-A3RD
1,TCGA-VS-A8QC-01A,1768,Primary Tumor,20.0,20.0,0.0,70.0,40.0,25.0,70.0,19006,female,None,white,CESC,Cervical squamous cell carcinoma and endocervi...,Cervix,NaN,TCGA-VS-A8QC
2,TCGA-IS-A3KA-01A,2721,Primary Tumor,0.0,0.0,0.0,85.0,0.0,20.0,80.0,26775,female,None,white,SARC,Sarcoma,Soft Tissue,NaN,TCGA-IS-A3KA
3,TCGA-HT-A5R5-01A,194,Primary Tumor,0.0,0.0,5.0,70.0,0.0,45.0,50.0,12403,female,None,white,LGG,Brain Lower Grade Glioma,Brain,NaN,TCGA-HT-A5R5
4,TCGA-73-4668-11A,None,Solid Tissue Normal,None,None,None,None,None,None,None,24255,female,None,american indian or alaska native,LUAD,Lung adenocarcinoma,Lung,2.0,TCGA-73-4668


In [16]:
# Survival
my_survival_df = tcga_survival_df[['case_submitter_id', 'vital_status', 'days_to_death', 'days_to_last_follow_up']]
my_survival_df['died'] = my_survival_df['vital_status'] == 'Dead'
my_survival_df['time'] = my_survival_df['days_to_death']
my_survival_df.replace("'--", None, inplace=True)
my_survival_df['time'].fillna(my_survival_df['days_to_last_follow_up'], inplace=True)
my_survival_df.drop(columns=['vital_status', 'days_to_death', 'days_to_last_follow_up'], inplace=True)
my_survival_df.drop_duplicates(inplace=True)
my_survival_df.rename({'case_submitter_id': 'submitter_id'}, axis=1, inplace=True)

my_survival_df.to_csv(os.path.join(out_dir, 'survival.csv'), index=False)
my_survival_df.head()

/var/folders/_6/468mk1cx3cb080z2r5gq0k_r0000gn/T/ipykernel_80740/925774716.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  my_survival_df['time'].fillna(my_survival_df['days_to_last_follow_up'], inplace=True)


,submitter_id,died,time
0,TCGA-E2-A1IU,False,None
3,TCGA-D1-A17N,False,None
7,TCGA-HS-A5NA,False,None
9,TCGA-BG-A0M6,True,671
19,TCGA-FA-A7Q1,True,248


In [17]:
# Gene expression
gene_names = np.loadtxt('data/cancer_genes.txt', dtype=str)

def read_rnaseq(sample_id, count_type = 'fpkm_uq_unstranded'):
    # count_type = unstranded, stranded_first, stranded_second, tpm_unstranded, fpkm_unstranded, fpkm_uq_unstranded
    sample_file_df = tcga_file_df[tcga_file_df['Sample ID'] == sample_id]
    if len(sample_file_df) < 1:
        return
    if len(sample_file_df) > 1:
        display(sample_file_df)
        raise ValueError(f"{sample_id} has multiple files")
    file_id = sample_file_df['File ID'].item()
    file_name = sample_file_df['File Name'].item()
    filepath = os.path.join(in_dir, 'rnaseq', file_id, file_name)
    df = pd.read_csv(filepath, sep='\t', header=1)
    df = df.drop_duplicates(subset='gene_name', keep='last')
    df = df[df['gene_name'].isin(gene_names)]
    df = df[['gene_name', count_type]]
    df[count_type] = 1e4 / df[count_type].sum() * df[count_type]
    df[count_type] = np.log10(df[count_type] + 1e-3)
    df.index = df['gene_name'].values
    df = df.drop(columns=['gene_name'])
    df = df.T.reset_index()
    df = df.drop(columns=['index'])
    df['sample_id'] = sample_id
    df = df[['sample_id'] + df.columns.tolist()[:-1]]
    return df

expression_rows = []
for i, sample_id in my_sample_df['sample_id'].items():
    print(len(my_sample_df), i, end='\r')
    expression_rows.append(read_rnaseq(sample_id))
my_expression_df = pd.concat(expression_rows)
my_expression_df.drop_duplicates(inplace=True)
my_expression_df.to_csv(os.path.join(out_dir, 'transcriptomic_features.csv'), index=False)

my_expression_df.head()

,sample_id,LASP1,HOXA11,CREBBP,ETV1,CCL26,CD79B,PAX7,BTK,BRCA1,...,NUTM2D,AC129492.1,DDX3X,FANCG,SSX2,ETV5,CEBPA,LSM14A,CUX1,AC124319.1
0,TCGA-GC-A3RD-01A,1.631206,0.594276,0.740795,-0.454943,-0.967093,-1.332186,-2.410469,-0.416861,0.333888,...,-1.114388,-1.660645,1.290988,-1.065987,-1.877241,-0.148462,1.070966,1.608954,0.770168,-0.460638
0,TCGA-VS-A8QC-01A,1.244755,-0.248246,0.818007,-0.222509,-0.289344,-0.410533,-2.233816,-0.282708,0.388120,...,-1.028403,-2.014316,1.246469,-0.323641,-3.000000,-0.239304,0.829953,1.364571,0.636477,-0.238899
0,TCGA-IS-A3KA-01A,1.100890,1.694080,0.659927,-0.328987,-1.494367,0.171299,-3.000000,0.688315,0.006311,...,-1.263271,-3.000000,0.856140,-0.274520,-3.000000,-0.245357,0.767333,1.499184,0.792272,-0.695466
0,TCGA-HT-A5R5-01A,1.366262,-3.000000,0.927563,1.046809,-0.959714,-1.420139,-1.143797,0.471621,-0.027070,...,-0.408522,-1.363434,1.348624,-1.137194,-3.000000,0.705336,0.889594,1.333906,1.217603,-3.000000
0,TCGA-73-4668-01A,1.556327,-2.005457,0.803671,0.559911,-0.405354,-0.241000,-3.000000,-0.032704,0.020101,...,-1.604152,-1.641368,1.160530,-0.958388,-3.000000,1.110087,0.407813,1.290540,0.586151,-0.125518
